# GRM shard processing -- binned phenotype covariance

Uses `GRM-pairs/grm_bin_sharded/grm_shard_tool` (a C++ CLI, not a Python
library -- shelled out to via `subprocess`) to bin the sharded GRM by
relatedness and compute the phenotype cross-product (covariance) within
each bin, for whichever phenotypes you pick.

**Designed to run before all shards are finished.** `grm_shard_tool
accumulate` operates one shard at a time and `merge` just sums whatever
accumulator files exist -- this notebook globs for whatever `.grm.bin.k`
files are actually present under `grm_shard_run.ipynb`'s persist
destination right now, uses only those, and prints how many of the total
it found. Rerunning later, as more shards finish, automatically picks up
more coverage and produces a more complete/precise estimate -- no code
changes needed, just rerun.

Plink-shard-reading only -- doesn't build or run `grm_shard_run.ipynb`/
`grm_shard_batch_submit.ipynb` itself, just consumes whatever shards they've
produced so far.

## Setup

Builds `grm_shard_tool` (idempotent -- `make` no-ops if already built).

In [ ]:
import glob
import os
import re
import subprocess

import pandas as pd
import matplotlib.pyplot as plt

REPO_DIR = os.path.expanduser("~/repos/aou-covariance")  # adjust if your checkout lives elsewhere
TOOL_DIR = f"{REPO_DIR}/GRM-pairs/grm_bin_sharded"
TOOL_BIN = f"{TOOL_DIR}/grm_shard_tool"

subprocess.run(["make"], cwd=TOOL_DIR, check=True)

print("REPO_DIR:", REPO_DIR)
print("TOOL_BIN:", TOOL_BIN, "exists:", os.path.isfile(TOOL_BIN))
assert os.path.isfile(TOOL_BIN), "missing tool binary: " + TOOL_BIN

## Paths

`SHARDS_DIR` is `grm_shard_run.ipynb`'s persist destination -- flat
`grm_shard_{k}_of_{N_SHARDS}.grm.bin.{k}` filenames, one shared
`genome_wide.grm.id`. (Not `grm_shard_batch_submit.ipynb`'s destination --
that one has a `dsub` output-path quirk that puts everything under a
literal, unexpanded `shard_${K}_of_${N_SHARDS}/` folder; point `SHARDS_DIR`
there instead if you're processing Batch-produced shards.)

In [ ]:
N_SHARDS = 300   # must match grm_shard_run.ipynb's N_SHARDS

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"
SHARDS_DIR = f"{BUCKET_DIR}/grm_shards"
GRM_ID = f"{SHARDS_DIR}/genome_wide.grm.id"

assert os.path.isfile(GRM_ID), (
    f"missing {GRM_ID} -- run grm_shard_run.ipynb's driver + persist cells first "
    f"(at least one completed shard is needed for its .grm.id)"
)

available_shards = sorted(glob.glob(f"{SHARDS_DIR}/grm_shard_*_of_{N_SHARDS}.grm.bin.*"))
print(f"{len(available_shards)}/{N_SHARDS} shards available so far")
if len(available_shards) < N_SHARDS:
    print("Partial run -- coverage of all pairs is incomplete. Rerun this notebook "
          "later (no changes needed) as more shards finish for a fuller estimate.")

## FID check

`grm_shard_tool`'s pheno lookup keys on the full `(FID, IID)` pair.
`write_grm_pheno()` (`02_phenotype/scripts/local/residualize_lib.R`) sets
`FID = IID = person_id`, but plink's `.grm.id` commonly defaults `FID` to
`"0"` -- confirmed true for this panel (all 177,301 rows). Rather than
trust either convention, `build_aligned_pheno()` below rewrites each
phenotype file's `FID` from the real `.grm.id`, keyed on `IID`, so this
works regardless of which convention is actually in effect.

In [ ]:
grm_ids = pd.read_csv(GRM_ID, sep=r"\s+", header=None, names=["FID", "IID"], dtype=str)
print(grm_ids.head())
print(f"unique FID values: {grm_ids['FID'].nunique()} across {len(grm_ids)} rows")
id_map = dict(zip(grm_ids["IID"], grm_ids["FID"]))  # real FID per IID, from the GRM itself


def build_aligned_pheno(pheno_path, out_path):
    """Rewrite a residualized phenotype file's FID to match the real .grm.id, keyed on IID."""
    pheno = pd.read_csv(pheno_path, sep=r"\s+", dtype={"FID": str, "IID": str})
    pheno["FID"] = pheno["IID"].map(id_map)
    unmatched = pheno["FID"].isna().sum()
    if unmatched:
        print(f"  WARNING: {unmatched}/{len(pheno)} IIDs in {os.path.basename(pheno_path)} "
              f"not found in .grm.id -- dropping")
        pheno = pheno.dropna(subset=["FID"])
    pheno.to_csv(out_path, sep=" ", index=False, columns=["FID", "IID", "Y"])
    return out_path

## Phenotypes

Lists what's available in `02_phenotype`'s residualized output
(`<phenotype_name>__<transform>__<covariate_set>.pheno`) so you can pick
real filenames for `PHENOTYPES` below. Use `covariate_set ==
"base_pcs_zip3_ses"` (the full model) to match `phenotype_exploration.ipynb`'s
R²-based ranking, and whichever `transform` that ranking used
(`invnorm`, most likely).

In [ ]:
PHENO_DIR = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot/data/02_phenotype/residualized"
)
print(sorted(os.listdir(PHENO_DIR)))

In [ ]:
PHENOTYPES = {
    # "display_name": f"{PHENO_DIR}/<actual_filename>.pheno",
    "bmi": f"{PHENO_DIR}/bmi__invnorm__base_pcs_zip3_ses.pheno",
    "alt": f"{PHENO_DIR}/alt__invnorm__base_pcs_zip3_ses.pheno",
}

## Accumulate + merge

One `accumulate` call per available shard, per phenotype, then one
`merge` per phenotype. `BINS_FILE` reuses `GRM-pairs/full_grm_bin/bins.txt`'s
existing default bin edges rather than generating new ones.

In [ ]:
BINS_FILE = f"{REPO_DIR}/GRM-pairs/full_grm_bin/bins.txt"
NBLOCKS = 50
SEED = 1
WORK_DIR = os.path.expanduser("~/grm_pheno_cov")
os.makedirs(WORK_DIR, exist_ok=True)

assert os.path.isfile(BINS_FILE), "missing bins file: " + BINS_FILE


def run_for_phenotype(name, pheno_path):
    aligned = build_aligned_pheno(pheno_path, f"{WORK_DIR}/{name}_aligned.pheno")

    acc_files = []
    for shard_bin in available_shards:
        m = re.search(r"grm_shard_(\d+)_of_\d+\.grm\.bin\.\d+$", shard_bin)
        k = int(m.group(1))
        acc_out = f"{WORK_DIR}/{name}_shard{k}.acc.tsv"
        subprocess.run(
            [TOOL_BIN, "accumulate",
             "--grm-id", GRM_ID, "--shard", shard_bin,
             "--parallel", str(k), str(N_SHARDS),
             "--pheno", aligned, "--bins", BINS_FILE,
             "--nblocks", str(NBLOCKS), "--seed", str(SEED),
             "--out", acc_out],
            check=True,
        )
        acc_files.append(acc_out)

    acc_list = f"{WORK_DIR}/{name}_acc_list.txt"
    with open(acc_list, "w") as f:
        f.write("\n".join(acc_files) + "\n")

    out_prefix = f"{WORK_DIR}/{name}_merged"
    subprocess.run(
        [TOOL_BIN, "merge", "--acc-list", acc_list, "--bins", BINS_FILE,
         "--nblocks", str(NBLOCKS), "--out-prefix", out_prefix],
        check=True,
    )
    return pd.read_csv(f"{out_prefix}.full.tsv", sep="\t")


results = {name: run_for_phenotype(name, path) for name, path in PHENOTYPES.items()}

## Plot -- covariance vs relatedness

Classic GREML-style diagnostic: x-axis is the GRM relatedness bin
midpoint, y-axis is the mean phenotype cross-product (covariance)
within that bin, error bars from the delete-block jackknife SE. The
title reports how many of the total shards actually went into this --
treat as a partial preview, not a final estimate, until it reads
`N_SHARDS/N_SHARDS`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, df in results.items():
    d = df[df["full_n"] > 0]
    ax.errorbar(d["bin_midpoint"], d["full_mean"], yerr=d["jk_se"],
                label=name, marker="o", ms=3, lw=1)
ax.axvline(0, color="grey", lw=0.5)
ax.set_xlabel("GRM relatedness (bin midpoint)")
ax.set_ylabel("Mean phenotype cross-product (covariance)")
ax.set_title(f"Covariance vs relatedness ({len(available_shards)}/{N_SHARDS} shards)")
ax.legend()
plt.tight_layout()
plt.show()

## Next steps

- Rerun this whole notebook later (no code changes) as more shards finish
  -- `available_shards` picks up whatever's newly present, and the
  `accumulate`/`merge` results get more complete/precise automatically.
- Once `grm_shard_run.ipynb` reports all `N_SHARDS` persisted, a run of
  this notebook is the final, full-coverage result -- worth a final rerun
  even if you've already looked at partial results along the way.
- Add more phenotypes to `PHENOTYPES` as needed; each is independent
  (its own `accumulate`/`merge` pass), so partial results for some
  phenotypes and full results for others can coexist across reruns.